# 🎮 Game 3: Calculating the ε-Gradient

## 📐 From Intuition to Mathematics

In Game 2, you used the **local slope** to navigate toward the minimum. You could *see* which direction was downhill.

**Now:** Let's learn how to **calculate** that slope mathematically!

## 🔬 The ε-Gradient Formula

To measure the slope at position `w_t`, we sample the loss at two nearby points:

### **Step-by-Step:**

1. **Choose a small window size** ε (epsilon)
   - Common values: ε = 0.1, 0.01, or 0.001

2. **Sample two points** around w_t:
```
   l = w_t - ε/2    (left point)
   r = w_t + ε/2    (right point)
```

3. **Evaluate loss** at both points:
```
   a = L(l)    (height at left)
   b = L(r)    (height at right)
```

4. **Calculate the slope** (rise over run):
```
   ∇_ε L(w_t) = (b - a) / ε
```

### **Visual Understanding:**
```
         b •────
          ╱│     
         ╱ │ b-a  ← rise
        ╱  │     
     a •───┴────
       ├───┤
         ε       ← run
   
   slope = rise/run = (b-a)/ε
```

### **Why This Works:**

- If `b > a`: Slope is **positive** → function increases to the right → move **left**
- If `b < a`: Slope is **negative** → function decreases to the right → move **right**
- If `b ≈ a`: Slope is **near zero** → you're at or near the **minimum**!

**Remember:** We move in the **opposite direction** of the slope (downhill, not uphill!)

---

## 🎯 Your Task

You'll implement the ε-gradient calculation and see it in action!

**Ready to code?** 🚀

In [1]:
#@title 🔧 Game 3 Setup Code (ε-Gradient Calculator) { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Button, VBox, HBox, Output, HTML, FloatText
from IPython.display import display, clear_output

class Game3_EpsilonGradient:
    def __init__(self):
        # Loss function (same as Game 2)
        self.L = lambda w: w**2 - 4*w + 7
        self.optimal_w = 2.0

        # Visualization state
        self.current_w = 0.0
        self.epsilon = 0.5
        self.output = Output()

    def calculate_epsilon_gradient(self, w, epsilon):
        """
        Calculate the ε-gradient at position w

        This is the formula students will implement!
        """
        l = w - epsilon/2
        r = w + epsilon/2
        a = self.L(l)
        b = self.L(r)
        gradient = (b - a) / epsilon
        return gradient, l, r, a, b

    def create_ui(self):
        """Create the interactive visualization"""

        # Input fields
        self.position_input = FloatText(
            value=0.0,
            description='Position w:',
            step=0.1,
            style={'description_width': '100px'},
            layout={'width': '280px'}
        )

        self.epsilon_input = FloatText(
            value=0.5,
            description='Window ε:',
            step=0.1,
            style={'description_width': '100px'},
            layout={'width': '280px'}
        )

        # Button
        calculate_button = Button(
            description='📐 Calculate Gradient',
            button_style='',
            layout={'width': '180px', 'height': '40px'}
        )
        calculate_button.style.button_color = '#3b82f6'
        calculate_button.style.text_color = 'white'
        calculate_button.style.font_weight = 'bold'
        calculate_button.on_click(self.calculate_and_plot)

        # Info display
        self.info_display = HTML("""
        <div style='background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #475569;'>
            <p style='color: #94a3b8; margin: 5px 0;'>Enter a position and window size, then click "Calculate Gradient"</p>
        </div>
        """)

        # Layout
        input_box = HBox([self.position_input, self.epsilon_input, calculate_button],
                        layout={'margin': '10px 0', 'align_items': 'center'})

        ui = VBox([
            self.output,
            input_box,
            self.info_display
        ])

        display(ui)

        # Show initial plot
        self.plot_initial()

    def plot_initial(self):
        """Show initial crater"""
        with self.output:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0a0e27')
            ax.set_facecolor('#1a1a2e')

            w_range = np.linspace(-1, 5, 200)
            loss_range = self.L(w_range)

            ax.plot(w_range, loss_range, color='#8b8b8b', linewidth=3.5,
                   label='Crater Surface')
            ax.fill_between(w_range, loss_range, 2, alpha=0.3, color='#3a3a3a')

            # Mark optimal
            ax.scatter([self.optimal_w], [self.L(self.optimal_w)],
                      color='#22c55e', s=300, marker='*',
                      edgecolors='#16a34a', linewidths=3, zorder=6,
                      label='Optimal Position')

            ax.text(0.5, 0.5, 'Enter position and click "Calculate Gradient"',
                   transform=ax.transAxes, fontsize=16, color='#94a3b8',
                   ha='center', va='center',
                   bbox=dict(boxstyle='round,pad=1', facecolor='#1e293b',
                            edgecolor='#475569', linewidth=2))

            ax.set_xlabel('Position (w)', fontsize=13, color='#c7d2fe', fontweight='bold')
            ax.set_ylabel('Loss L(w)', fontsize=13, color='#c7d2fe', fontweight='bold')
            ax.set_title('ε-Gradient Calculator', fontsize=16, color='#e0e7ff',
                        fontweight='bold', pad=20)
            ax.grid(True, alpha=0.15, color='#4a5568')
            ax.legend(fontsize=11, facecolor='#1a1a2e', edgecolor='#4a4a6a',
                     labelcolor='#e0e7ff')
            ax.set_xlim(-1, 5)
            ax.set_ylim(2, max(loss_range) + 1)

            for spine in ax.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax.tick_params(colors='#a5b4fc')

            plt.tight_layout()
            plt.show()

    def calculate_and_plot(self, b):
        """Calculate gradient and visualize"""
        try:
            w = float(self.position_input.value)
            epsilon = float(self.epsilon_input.value)
        except ValueError:
            with self.output:
                clear_output(wait=True)
                print("⚠️ Please enter valid numbers")
            return

        if epsilon <= 0:
            with self.output:
                clear_output(wait=True)
                print("⚠️ Window size ε must be positive")
            return

        # Calculate gradient
        gradient, l, r, a, b = self.calculate_epsilon_gradient(w, epsilon)

        # Plot
        self.plot_gradient(w, epsilon, gradient, l, r, a, b)

        # Update info
        self.update_info(w, epsilon, gradient, l, r, a, b)

    def plot_gradient(self, w, epsilon, gradient, l, r, a, b):
        """Visualize the ε-gradient calculation"""
        with self.output:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0a0e27')
            ax.set_facecolor('#1a1a2e')

            # Plot full curve
            w_range = np.linspace(-1, 5, 200)
            loss_range = self.L(w_range)
            ax.plot(w_range, loss_range, color='#8b8b8b', linewidth=3,
                   label='Crater Surface', alpha=0.6)
            ax.fill_between(w_range, loss_range, 2, alpha=0.2, color='#3a3a3a')

            # Highlight the ε-window
            window_w = np.linspace(l, r, 50)
            window_loss = self.L(window_w)
            ax.plot(window_w, window_loss, color='#60a5fa', linewidth=5,
                   label=f'ε-Window (ε={epsilon})', zorder=4)

            # Mark the two sample points
            ax.scatter([l], [a], color='#fbbf24', s=200, marker='o',
                      edgecolors='#f59e0b', linewidths=3, zorder=6,
                      label=f'Left: L({l:.2f})={a:.2f}')
            ax.scatter([r], [b], color='#fbbf24', s=200, marker='o',
                      edgecolors='#f59e0b', linewidths=3, zorder=6,
                      label=f'Right: L({r:.2f})={b:.2f}')

            # Draw the secant line (approximation of slope)
            ax.plot([l, r], [a, b], color='#ef4444', linewidth=3,
                   linestyle='--', label='Secant Line', zorder=5)

            # Current position marker
            ax.scatter([w], [self.L(w)], color='#ef4444', s=300, marker='v',
                      edgecolors='#dc2626', linewidths=3, zorder=7,
                      label=f'Position w={w:.2f}')

            # Visual annotation for rise and run
            # Rise (vertical line)
            ax.plot([r, r], [a, b], color='#22c55e', linewidth=2.5,
                   linestyle=':', alpha=0.7, zorder=5)
            ax.text(r + 0.15, (a + b)/2, f'rise\n{b-a:.2f}',
                   fontsize=11, color='#22c55e', fontweight='bold',
                   va='center')

            # Run (horizontal line)
            ax.plot([l, r], [a, a], color='#a78bfa', linewidth=2.5,
                   linestyle=':', alpha=0.7, zorder=5)
            ax.text((l + r)/2, a - 0.3, f'run = ε = {epsilon:.2f}',
                   fontsize=11, color='#a78bfa', fontweight='bold',
                   ha='center')

            # Gradient result box (NO ARROW)
            if gradient > 0:
                direction_text = "↗️ Positive (uphill to right)"
                box_color = '#fbbf24'
            elif gradient < 0:
                direction_text = "↘️ Negative (downhill to right)"
                box_color = '#60a5fa'
            else:
                direction_text = "➡️ Zero (flat)"
                box_color = '#94a3b8'

            # Place result box at top
            ax.text(0.5, 0.95, f'∇_ε L = {gradient:.3f}\n{direction_text}',
                   transform=ax.transAxes,
                   fontsize=13, fontweight='bold', color='#fbbf24',
                   ha='center', va='top',
                   bbox=dict(boxstyle='round,pad=0.8', facecolor='#1e40af',
                            edgecolor='#60a5fa', linewidth=2, alpha=0.9))

            # Optimal marker
            ax.scatter([self.optimal_w], [self.L(self.optimal_w)],
                      color='#22c55e', s=300, marker='*',
                      edgecolors='#16a34a', linewidths=3, zorder=6,
                      label='Optimal Position')

            ax.set_xlabel('Position (w)', fontsize=13, color='#c7d2fe', fontweight='bold')
            ax.set_ylabel('Loss L(w)', fontsize=13, color='#c7d2fe', fontweight='bold')
            ax.set_title('ε-Gradient Visualization: How the Slope is Calculated',
                        fontsize=16, color='#e0e7ff', fontweight='bold', pad=20)
            ax.grid(True, alpha=0.15, color='#4a5568')
            ax.legend(fontsize=10, loc='upper right', facecolor='#1a1a2e',
                     edgecolor='#4a4a6a', labelcolor='#e0e7ff', framealpha=0.9)
            ax.set_xlim(-1, 5)
            ax.set_ylim(2, max(loss_range) + 1)

            for spine in ax.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax.tick_params(colors='#a5b4fc')

            plt.tight_layout()
            plt.show()

    def update_info(self, w, epsilon, gradient, l, r, a, b):
        """Update information display"""

        # Determine interpretation
        if abs(gradient) < 0.1:
            interpretation = "Nearly flat - close to a minimum or maximum!"
            color = '#14532d'
            border = '#22c55e'
        elif gradient > 0:
            interpretation = "Positive slope - function increases to the right. Move LEFT to descend."
            color = '#1e293b'
            border = '#fbbf24'
        else:
            interpretation = "Negative slope - function decreases to the right. Move RIGHT to descend."
            color = '#1e293b'
            border = '#60a5fa'

        self.info_display.value = f"""
        <div style='background: {color}; padding: 15px; border-radius: 8px; border: 1px solid {border};'>
            <h4 style='color: #e0e7ff; margin-top: 0;'>📊 Calculation Results</h4>
            <p style='color: #e2e8f0; margin: 5px 0; font-family: monospace;'>
                <strong>Position:</strong> w = {w:.2f}<br>
                <strong>Window:</strong> ε = {epsilon:.2f}<br>
                <strong>Left point:</strong> l = {w:.2f} - {epsilon/2:.2f} = {l:.2f}<br>
                <strong>Right point:</strong> r = {w:.2f} + {epsilon/2:.2f} = {r:.2f}<br>
                <strong>Left loss:</strong> a = L({l:.2f}) = {a:.2f}<br>
                <strong>Right loss:</strong> b = L({r:.2f}) = {b:.2f}<br>
                <strong>Rise:</strong> b - a = {b:.2f} - {a:.2f} = {b-a:.2f}<br>
                <strong>Run:</strong> ε = {epsilon:.2f}<br>
                <strong>Gradient:</strong> ∇_ε L(w) = {b-a:.2f} / {epsilon:.2f} = <span style='color: #fbbf24; font-size: 18px;'>{gradient:.4f}</span>
            </p>
            <p style='color: #fbbf24; margin: 10px 0 5px 0; font-size: 15px;'>
                <strong>💡 Interpretation:</strong> {interpretation}
            </p>
        </div>
        """

print("✅ Game 3 setup complete! Ready to calculate gradients.")

✅ Game 3 setup complete! Ready to calculate gradients.


In [2]:
#@title ▶️ Launch Game 3: ε-Gradient Calculator { display-mode: "form" }
#@markdown Try different positions and window sizes to see how the gradient is calculated!

game3 = Game3_EpsilonGradient()
game3.create_ui()

## 💻 Your Turn: Implement the ε-Gradient

Now that you've seen how the ε-gradient works visually, it's time to **implement it yourself**!

### 📝 Your Task

Complete the `calculate_epsilon_gradient` function below by filling in the missing code.

**Remember the steps:**
1. Calculate left point: `l = w_t - ε/2`
2. Calculate right point: `r = w_t + ε/2`
3. Evaluate loss at both points: `a = L(l)` and `b = L(r)`
4. Calculate gradient: `∇_ε L(w_t) = (b - a) / ε`

### 🎯 Instructions

1. Replace each `None` with the correct code
2. Run the cell to test your implementation
3. All tests should show ✓ when your code is correct

**Ready? Edit the code below and run it!** 🚀

In [3]:
# 💻 CODING EXERCISE: Implement the ε-Gradient
# Replace each 'None' with the correct code

def calculate_epsilon_gradient(loss_fn, w_t, epsilon):
    """
    Calculate the ε-gradient of loss_fn at position w_t

    Args:
        loss_fn: Function that takes w and returns L(w)
        w_t: Current position
        epsilon: Window size (ε)

    Returns:
        gradient: The ε-gradient ∇_ε L(w_t)
    """
    # Step 1: Calculate left and right points
    # l should be: w_t - epsilon/2
    # r should be: w_t + epsilon/2
    l = None  # ← Replace None with your code
    r = None  # ← Replace None with your code

    # Step 2: Evaluate loss at both points
    # a should be: loss_fn(l)
    # b should be: loss_fn(r)
    a = None  # ← Replace None with your code
    b = None  # ← Replace None with your code

    # Step 3: Calculate gradient (rise over run)
    # gradient should be: (b - a) / epsilon
    gradient = None  # ← Replace None with your code

    return gradient

In [4]:

# ============================================================
# TEST YOUR IMPLEMENTATION
# ============================================================

# Define the loss function (same as in the game)
L = lambda w: w**2 - 4*w + 7

# Test parameters
test_positions = [0.0, 1.0, 2.0, 3.0]
epsilon = 0.1
expected_gradients = [-4.0, -2.0, 0.0, 2.0]

# Run tests
print("=" * 60)
print("🧪 TESTING YOUR ε-GRADIENT IMPLEMENTATION")
print("=" * 60)
print()

try:
    print(f"Loss function: L(w) = w² - 4w + 7")
    print(f"Window size: ε = {epsilon}")
    print()
    print("Position (w) | Your Gradient | Expected  | Status")
    print("-" * 60)

    all_passed = True

    for i, w in enumerate(test_positions):
        try:
            your_grad = calculate_epsilon_gradient(L, w, epsilon)
            expected = expected_gradients[i]

            # Check if None was returned (not implemented)
            if your_grad is None:
                print(f"    {w:.1f}      |     None       |  {expected:6.2f}   |   ⚠️  Not implemented")
                all_passed = False
            else:
                # Check if close to expected value
                match = abs(your_grad - expected) < 0.2
                status = "✅ Pass" if match else "❌ Fail"
                print(f"    {w:.1f}      |    {your_grad:7.4f}    |  {expected:6.2f}   |   {status}")
                if not match:
                    all_passed = False
        except TypeError as e:
            print(f"    {w:.1f}      |     Error      |  {expected:6.2f}   |   ❌ TypeError")
            all_passed = False
        except Exception as e:
            print(f"    {w:.1f}      |     Error      |  {expected:6.2f}   |   ❌ {type(e).__name__}")
            all_passed = False

    print()
    print("=" * 60)

    if all_passed:
        print("🎉 CONGRATULATIONS! All tests passed!")
        print("✅ Your ε-gradient implementation is correct!")
        print()
        print("💡 Key Insights:")
        print("   • The gradient tells you the slope at a position")
        print("   • Negative gradient → move RIGHT to descend")
        print("   • Positive gradient → move LEFT to descend")
        print("   • Zero gradient → you're at a critical point!")
    else:
        print("⚠️  Some tests failed. Check your implementation:")
        print()
        print("🔍 Common Issues:")
        print("   1. Did you replace all 'None' with actual code?")
        print("   2. Check the formula: l = w_t - epsilon/2")
        print("   3. Check the formula: r = w_t + epsilon/2")
        print("   4. Check: a = loss_fn(l) and b = loss_fn(r)")
        print("   5. Check: gradient = (b - a) / epsilon")
        print()
        print("💡 Tip: Look at the visualization above to remind yourself of the formula!")

except Exception as e:
    print("❌ ERROR: Something went wrong!")
    print(f"   {type(e).__name__}: {e}")
    print()
    print("🔍 Debugging hints:")
    print("   • Make sure you replaced all 'None' values")
    print("   • Check for syntax errors")
    print("   • Look at the formula in the visualization above")

print("=" * 60)

🧪 TESTING YOUR ε-GRADIENT IMPLEMENTATION

Loss function: L(w) = w² - 4w + 7
Window size: ε = 0.1

Position (w) | Your Gradient | Expected  | Status
------------------------------------------------------------
    0.0      |     None       |   -4.00   |   ⚠️  Not implemented
    1.0      |     None       |   -2.00   |   ⚠️  Not implemented
    2.0      |     None       |    0.00   |   ⚠️  Not implemented
    3.0      |     None       |    2.00   |   ⚠️  Not implemented

⚠️  Some tests failed. Check your implementation:

🔍 Common Issues:
   1. Did you replace all 'None' with actual code?
   2. Check the formula: l = w_t - epsilon/2
   3. Check the formula: r = w_t + epsilon/2
   4. Check: a = loss_fn(l) and b = loss_fn(r)
   5. Check: gradient = (b - a) / epsilon

💡 Tip: Look at the visualization above to remind yourself of the formula!
